# 🚀 LiDAR: Lookahead Sample Reward Guidance Replication (Table 2)
### **Paper**: [Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models (ICML 2026 Spotlight)](https://arxiv.org/abs/2602.03211)
### **Target**: Reproduce Table 2 Setting: **SD v1.5 + LiDAR (DPM-5 / $n=50$)**

---
### 📊 Target Paper Results (Table 2):
| Backbone & Setting | Guidance Method | ImageReward (↑) | CLIP Score (↑) | HPS v2.1 (↑) | GenEval (↑) | Time (s/run) | Memory (GiB) |
| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **SD v1.5 (w/ DDPM 100 steps)** | **LiDAR (DPM-5 / $n=50$)** | **0.384** | **0.278** | **0.276** | **0.478** | 13.41s | 8.90 GiB |
| **SD v1.5 (w/ DDIM 50 steps)** | **LiDAR (DPM-5 / $n=50$)** | **0.378** | **0.278** | **0.277** | **0.475** | 9.92s | 8.90 GiB |
| *Vanilla SD v1.5 (DDPM 100)* | Baseline | 0.001 | 0.271 | 0.263 | 0.426 | 7.07s | 8.90 GiB |
| *Vanilla SD v1.5 (DDIM 50)* | Baseline | -0.125 | 0.269 | 0.270 | 0.423 | 3.58s | 8.90 GiB |

---
### 🛠 Key Features of this Kaggle Notebook:
1. **Self-Contained & Automated**: Clones repository, installs dependencies, and prepares models seamlessly on Kaggle T4 / P100 / A100.
2. **Crash-Resilient & Resume Mechanism (Fallback)**: Both Phase 1 and Phase 2 automatically track completed prompts. If Kaggle disconnects or hits execution time limits, simply re-running the cells will resume immediately from where it stopped.
3. **Configurable Subsetting**: Run on the full GenEval dataset (553 prompts) or test on a small subset (e.g., 10 prompts) for quick validation.
4. **Auto-Packaging & Archiving**: Compresses latents, generated images, and result JSONs into `.zip` archives ready for 1-click download.


## 1. System & GPU Environment Verification


In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi


## 2. Setup Repository & Dependencies


In [ ]:
# Setup workspace directory in Kaggle
import os

WORKDIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(WORKDIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {WORKDIR}

%cd {WORKDIR}/Diffusion-LiDAR-Sampling

# Install dependencies (using https for ImageReward)
!pip install -q diffusers transformers accelerate ftfy timm einops hpsv2
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git


## 3. Experiment Configuration (Table 2 Parameters)
Key Parameters:
- **Phase 1 Lookahead Sampler**: DPM-Solver with $S=5$ steps (`--num_inference_steps=5`), $n=50$ particles (`--num_particles=50`).
- **Phase 2 Target Sampler**: SD 1.5 with 100 DDPM steps (`--num_inference_steps=100`, `--eta=1.0`) or 50 DDIM steps (`--num_inference_steps=50`, `--eta=0.0`).
- **Guidance Parameters**: Scale $s=12.5$ (`--scale=12.5`), $\lambda=5000$ (`--lmbda=5000`), early-stage guidance cutoff $T_{end}=200$ (`--resample_t_end=200`).
- **Evaluated Target Particles**: $N=4$ images per prompt as standard GenEval protocol.


In [ ]:
# ==================== EXPERIMENT HYPERPARAMETERS ====================
SEED = 100                       # Random seed (100 or 42)
NUM_LOOKAHEAD_PARTICLES = 50     # n = 50 lookahead particles
LOOKAHEAD_STEPS = 5              # DPM-5 solver steps
LOOKAHEAD_TAG = f"{SEED}_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"

# Target Sampling Parameters (SD v1.5 with DDPM 100 steps)
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
NUM_TARGET_STEPS = 100           # 100 for DDPM (or 50 for DDIM)
ETA = 1.0                        # 1.0 for DDPM (or 0.0 for DDIM)
TARGET_PARTICLES = 4             # 4 images per prompt (GenEval benchmark protocol)
SCALE = 12.5                     # Guidance scale s = 12.5 for SD v1.5
LAMBDA = 5000                    # Temperature lambda = 5000
RESAMPLE_T_END = 200             # Early stage denoising interval cutoff [1.0, 0.2]
TOP_K = 50                       # top-k lookaheads (50)

# Dataset & Prompt Limits
PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"
# Set MAX_PROMPTS = 553 for full benchmark, or e.g. 10-20 for rapid testing
MAX_PROMPTS = 553

RUN_NAME = f"LiDAR_SD15_DPM5_n50_steps{NUM_TARGET_STEPS}_seed{SEED}"

print(f"Run Name: {RUN_NAME}")
print(f"Lookahead Tag: {LOOKAHEAD_TAG}")
print(f"Total Prompts to Evaluate: {MAX_PROMPTS}")


## 4. Phase 1: Lookahead Sampling & Reward Annotation
Generates $n=50$ particles per prompt using 5-step DPM-Solver sampling, evaluating each with `ImageReward` (and `Clip-Score`).
- **Resume Protection**: If interrupted, re-running this cell automatically skips already computed prompts in `Lookahead_samples/{LOOKAHEAD_TAG}`.


In [ ]:
# Run Phase 1 Lookahead Sampling
lookahead_cmd = f"""python lookahead_sampling.py \
    --seed={SEED} \
    --num_particles={NUM_LOOKAHEAD_PARTICLES} \
    --num_inference_steps={LOOKAHEAD_STEPS} \
    --model_name="{MODEL_NAME}" \
    --prompt_path="{PROMPT_FILE}" \
    --max_prompt={MAX_PROMPTS} \
    --guidance_reward_fn="ImageReward" \
    --metrics_to_compute="ImageReward#Clip-Score" \
    --save_individual_images=True \
    --resume
"""

print(">>> Starting Phase 1 Lookahead Sampling...")
!{lookahead_cmd}


## 5. Phase 2: LiDAR Test-Time Steering Sampling
Performs reward-tilted diffusion sampling using closed-form LiDAR guidance toward high-reward lookaheads.
- **Resume Protection**: Automatically skips already completed target prompts.


In [ ]:
# Run Phase 2 LiDAR Sampling
lidar_cmd = f"""python LiDAR_sampling.py \
    --seed={SEED} \
    --model_name="{MODEL_NAME}" \
    --num_particles={TARGET_PARTICLES} \
    --num_inference_steps={NUM_TARGET_STEPS} \
    --eta={ETA} \
    --use_rag \
    --lookahead_path="{LOOKAHEAD_TAG}" \
    --top_k={TOP_K} \
    --scale={SCALE} \
    --lmbda={LAMBDA} \
    --resample_t_end={RESAMPLE_T_END} \
    --prompt_path="{PROMPT_FILE}" \
    --max_prompt={MAX_PROMPTS} \
    --guidance_reward_fn="ImageReward" \
    --metrics_to_compute="ImageReward#Clip-Score#Clip-Diversity#HumanPreference#AS" \
    --save_individual_images \
    --run_name="{RUN_NAME}" \
    --resume
"""

print(">>> Starting Phase 2 LiDAR Sampling...")
!{lidar_cmd}


## 6. Quantitative Evaluation vs Table 2 Baseline


In [ ]:
import json
import pandas as pd
from IPython.display import display

results_file = f"Target_samples/{RUN_NAME}/final_metrics.json"

if os.path.exists(results_file):
    with open(results_file, "r") as f:
        metrics = json.load(f)
    
    reproduced_metrics = {
        "ImageReward (IR)": f"{metrics.get('ImageReward', {}).get('mean', 0.0):.4f}",
        "CLIP Score": f"{metrics.get('Clip-Score', {}).get('mean', 0.0):.4f}",
        "HPS v2.1": f"{metrics.get('HumanPreference', {}).get('mean', 0.0):.4f}",
        "CLIP Diversity": f"{metrics.get('Clip-Diversity', {}).get('mean', 0.0):.4f}",
        "Aesthetic Score (AS)": f"{metrics.get('AS', {}).get('mean', 0.0):.4f}",
    }
    
    paper_target_ddpm = {
        "ImageReward (IR)": "0.384",
        "CLIP Score": "0.278",
        "HPS v2.1": "0.276",
        "CLIP Diversity": "-",
        "Aesthetic Score (AS)": "-"
    }

    paper_target_ddim = {
        "ImageReward (IR)": "0.378",
        "CLIP Score": "0.278",
        "HPS v2.1": "0.277",
        "CLIP Diversity": "-",
        "Aesthetic Score (AS)": "-"
    }

    df = pd.DataFrame([
        reproduced_metrics,
        paper_target_ddpm if NUM_TARGET_STEPS == 100 else paper_target_ddim
    ], index=["Reproduced Run", f"Paper Table 2 (DD{'PM-100' if NUM_TARGET_STEPS==100 else 'IM-50'})"])
    
    print("
=================== 📊 QUANTITATIVE METRICS COMPARISON ===================")
    display(df)
else:
    print(f"Results file not found at {results_file}. Run Phase 2 first.")


## 7. Qualitative Generated Sample Inspection


In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

target_dir = f"Target_samples/{RUN_NAME}"
grid_images = sorted(glob.glob(f"{target_dir}/*/grid.png"))

if grid_images:
    print(f"Found {len(grid_images)} generated prompt grids. Displaying first 3:")
    for img_path in grid_images[:3]:
        img = Image.open(img_path)
        plt.figure(figsize=(16, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Prompt Index: {os.path.basename(os.path.dirname(img_path))}")
        plt.show()
else:
    print("No sample grids found yet.")


## 8. Export & Archive Results (Single-Click Download)
Compress the results and lookahead folders into zip archives in `/kaggle/working` for easy downloading.


In [ ]:
output_zip_target = f"/kaggle/working/{RUN_NAME}_results.zip"
output_zip_lookahead = f"/kaggle/working/lookahead_{LOOKAHEAD_TAG}.zip"

if os.path.exists(f"Target_samples/{RUN_NAME}"):
    !zip -q -r {output_zip_target} Target_samples/{RUN_NAME}
    print(f"✅ Target samples zipped: {output_zip_target} ({os.path.getsize(output_zip_target) / (1024*1024):.2f} MB)")

if os.path.exists(f"Lookahead_samples/{LOOKAHEAD_TAG}"):
    !zip -q -r {output_zip_lookahead} Lookahead_samples/{LOOKAHEAD_TAG}
    print(f"✅ Lookahead samples zipped: {output_zip_lookahead} ({os.path.getsize(output_zip_lookahead) / (1024*1024):.2f} MB)")

print("
🎉 All outputs packaged! You can download the zip files from the Kaggle Output explorer.")
